<a href="https://colab.research.google.com/github/Colanimmy/AAI2025-DEVIN-COPY/blob/2026fall/Coding_Exercise_ML_Basics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# https://www.kaggle.com/datasets/harlfoxem/housesalesprediction

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

df = pd.read_csv("kc_house_data.csv")
df = df[['sqft_living', 'zipcode', 'price']]
df = df.rename(columns={'sqft_living': 'square_footage', 'zipcode': 'location'})

X = df[['square_footage', 'location']]
y = df['price']

preprocessor = ColumnTransformer(
transformers=[
('location', OneHotEncoder(sparse_output=False), ['location'])
], remainder='passthrough')

model = Pipeline(steps=[
('preprocessor', preprocessor),
('regressor', LinearRegression())
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
random_state=42)

model.fit(X_train, y_train)

new_house = pd.DataFrame({'square_footage': [2000], 'location': [98103]})
predicted_price = model.predict(new_house)
print(f"Predicted price for a 2000 sq ft house in ZIP code 98103: ${predicted_price[0]:,.2f}")

feature_names = ((model.named_steps['preprocessor']
.named_transformers_['location']
.get_feature_names_out(['location'])).tolist() +
['square_footage'])
coefficients = model.named_steps['regressor'].coef_
print("\nModel Coefficients:")
for feature, coef in zip(feature_names, coefficients):
    print(f"{feature}: {coef:.2f}")

In [ ]:
# https://www.kaggle.com/datasets/blastchar/telco-customer-churn

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')

df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df = df.dropna(subset=['TotalCharges'])
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

X = df[['tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen',
        'InternetService']]
y = df['Churn']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['tenure', 'MonthlyCharges', 'TotalCharges',
                                   'SeniorCitizen']),
        ('cat', OneHotEncoder(sparse_output=False), ['InternetService'])
    ])

model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(random_state=42))
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
                                                    random_state=42)

model.fit(X_train, y_train)

new_customer = pd.DataFrame({
    'tenure': [12],
    'MonthlyCharges': [70],
    'TotalCharges': [840],
    'SeniorCitizen': [0],
    'InternetService': ['Fiber optic']
})

churn_probability = model.predict_proba(new_customer)[0][1]  # Probability of churn (class 1)

threshold = 0.5
churn_prediction = 1 if churn_probability > threshold else 0

print(f"Churn Probability for new customer: {churn_probability:.2f}")
print(f"Churn Prediction (1 = churn, 0 = no churn): {churn_prediction}")

feature_names = ['tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen'] + \
    (model.named_steps['preprocessor']
     .named_transformers_['cat']
     .get_feature_names_out(['InternetService'])).tolist()

coefficients = model.named_steps['classifier'].coef_[0]

print("\nModel Coefficients:")
for feature, coef in zip(feature_names, coefficients):
    print(f"{feature}: {coef:.2f}")

In [ ]:
# https://www.kaggle.com/datasets/vjchoudhary7/customer-segmentation-tutorial-in-python?utm_source=chatgpt.com

import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

df = pd.read_csv('Mall_Customers.csv')

df = df.rename(columns={
    'Annual Income (k$)': 'annual_spending',
    'Spending Score (1-100)': 'purchase_frequency',
    'Age': 'age',
    'Gender': 'region'
})

features = ['annual_spending', 'purchase_frequency', 'age']
X = df[features]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

inertia = []
K = range(1, 6)
for k in K:
    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(X_scaled)
    inertia.append(kmeans.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(K, inertia, 'bo-')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia')
plt.title('Elbow Method for Optimal K')
plt.savefig('elbow_plot.png')
plt.close()

optimal_k = 3
kmeans = KMeans(n_clusters=optimal_k, random_state=42)
df['cluster'] = kmeans.fit_predict(X_scaled)

cluster_summary = df.groupby('cluster')[features].mean().round(2)
print("Cluster Characteristics:")
print(cluster_summary)

for cluster in range(optimal_k):
    print(f"\nCluster {cluster} Strategy:")
    if cluster_summary.loc[cluster, 'annual_spending'] > 70:
        print("High-spending customers: Offer exclusive promotions or loyalty rewards.")
    elif cluster_summary.loc[cluster, 'purchase_frequency'] > 60:
        print("Frequent buyers: Provide bulk discounts or subscription plans.")
    else:
        print("Low-engagement customers: Send personalized re-engagement campaigns.")

df.to_csv('customer_segments.csv', index=False)